# pAIr 추론 서버 - RunPod 에서 실제 모델로 띄워 보기

**이 노트북 하나로** 코드 받기 → 라이브러리 설치 → 모델 5개 내려받기·검증 → 서버 실행 → GPU 확인 → 재생 시험까지 합니다.
위에서부터 한 칸씩 `Shift + Enter` 로 실행하세요. (RunPod 의 PyTorch 템플릿 + GPU 1장 Pod 기준, JupyterLab 에서 열기)

- 사진·영상은 **포함되어 있지 않습니다**(응시자·개인 사진은 올리지 않는다). 사진이 없으면 합성 이미지로 속도만 잽니다.
- 끝나면 **Pod 를 Stop/Terminate** 하세요. 켜 두면 과금됩니다.
- 이 노트북은 GPU 에서 **아직 한 번도 돌려 보지 못했습니다.** 막히는 칸이 있으면 그 칸의 출력과 함께 알려 주세요.

## 1. 설정 · GPU 확인

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# 코드 저장소. 비공개라면 아래 REPO_TOKEN 칸에 GitHub 토큰(읽기 전용)을 입력한다. 토큰은 화면·파일에 남기지 않는다.
REPO_URL = "https://github.com/KosAmigo/pAIr-vision.git"
REPO_BRANCH = "main"

WORK = Path("/workspace") if Path("/workspace").exists() else Path.home()
ROOT = WORK / "pAIr-vision"
MODELS = ROOT / "models"
print("작업 폴더:", ROOT)

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "nvidia-smi 없음 (GPU 가 안 잡힌 Pod 일 수 있음)")
print("python:", sys.version.split()[0])

## 2. 코드 받기
공개 저장소면 그대로 실행됩니다. 비공개면 토큰을 물어봅니다(입력은 화면에 보이지 않음).

In [ ]:
import getpass

if (ROOT / ".git").exists():
    print("이미 있음 -> 최신으로 갱신")
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=False)
else:
    url = REPO_URL
    probe = subprocess.run(["git", "ls-remote", url, REPO_BRANCH], capture_output=True, text=True,
                           env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if probe.returncode != 0:
        token = getpass.getpass("저장소가 비공개입니다. GitHub 토큰(읽기 전용)을 붙여넣으세요: ")
        url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
    result = subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", url, str(ROOT)],
                            capture_output=True, text=True)
    # 토큰이 출력·.git/config 에 남지 않게 정리
    subprocess.run(["git", "-C", str(ROOT), "remote", "set-url", "origin", REPO_URL], check=False)
    print(result.stdout[-300:], result.stderr[-300:].replace(url, "<url>"))
    assert result.returncode == 0, "git clone 실패"
os.chdir(ROOT)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. 라이브러리 설치
RunPod PyTorch 템플릿에 torch 가 이미 있으면 그대로 두고 나머지(서버·transformers·mediapipe 등)만 설치합니다. 몇 분 걸립니다.

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[real]", "huggingface_hub"],
                   capture_output=True, text=True)
print(r.stdout[-500:], r.stderr[-800:])
assert r.returncode == 0, "pip 설치 실패"

import torch
print("torch", torch.__version__, "| CUDA 사용 가능:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GPU 없음")

## 4. 모델 5개 내려받기 · 검증
- D-FINE-s(`ustc-community/dfine-small-coco`), OWLv2(`google/owlv2-base-patch16-ensemble`) 는 Hugging Face 에서,
  MediaPipe 손·얼굴·포즈(`.task` 3개)는 Google 저장소에서 받습니다. 합쳐서 약 700MB.
- 받은 뒤 `configs/models.lock` 의 sha256 과 대조합니다. **다르면 멈춥니다**(저장소의 모델 파일이 바뀐 것이므로 이유를 확인한 뒤에만 계속).

In [ ]:
import urllib.request
from huggingface_hub import hf_hub_download

from pair_vision.models.lock import ModelLockError, load_lock, verify_lock

MODELS.mkdir(exist_ok=True)
locked = load_lock(ROOT / "configs" / "models.lock")

# 단일 파일 모델(.task) 은 lock 의 repo 가 곧 내려받는 주소다
for m in locked:
    if m.directory.endswith(".task"):
        target = MODELS / m.directory
        if not target.exists():
            print("받는 중:", m.directory)
            urllib.request.urlretrieve(m.repo, target)
# Hugging Face 모델은 lock 에 적힌 파일만 받는다
for m in locked:
    if not m.directory.endswith(".task"):
        for rel in m.files:
            if not (MODELS / m.directory / rel).exists():
                print("받는 중:", f"{m.repo}/{rel}")
                hf_hub_download(m.repo, rel, local_dir=MODELS / m.directory)

try:
    verify_lock(MODELS, locked)
    print("모델 5개 sha256 검증 통과")
except ModelLockError as exc:
    print(exc)
    raise

## 5. 서버 실행
API 키·콜백 비밀은 물어봅니다(화면에 보이지 않음). 콜백 주소를 비워 두면 이벤트를 보내지 않고 건수만 셉니다(시험용).

In [ ]:
import getpass
import time
import urllib.request as ur

API_KEY = getpass.getpass("서버 API 키(임의 문자열, 시험용): ") or "dev-key"
CALLBACK_URL = input("백엔드 콜백 주소(없으면 그냥 Enter): ").strip()
env = {**os.environ, "VISION_API_KEY": API_KEY, "VISION_DETECTOR": "real", "VISION_MODELS_DIR": str(MODELS),
       "PYTHONPATH": str(ROOT / "src")}
if CALLBACK_URL:
    env["VISION_CALLBACK_URL"] = CALLBACK_URL
    env["VISION_CALLBACK_SECRET"] = getpass.getpass("콜백 서명 비밀: ")

log = open(WORK / "vision-server.log", "w")
server = subprocess.Popen([sys.executable, "-m", "pair_vision"], cwd=ROOT, env=env, stdout=log, stderr=subprocess.STDOUT)

import json
ready = None
for _ in range(60):  # 모델 적재(sha256 검증 포함)에 수십 초
    time.sleep(3)
    if server.poll() is not None:
        break
    try:
        req = ur.Request("http://localhost:8000/readyz", headers={"Authorization": f"Bearer {API_KEY}"})
        ready = json.load(ur.urlopen(req, timeout=3))
        break
    except Exception:
        continue
print(ready or "서버가 안 떴습니다. 로그 마지막 줄:\n" + "".join(open(WORK / "vision-server.log").readlines()[-15:]))
if ready:
    print("\n>>> gpu =", ready["gpu"], "(False 면 GPU 를 못 쓰는 중. 3번 칸의 torch 출력과 Pod 의 GPU 설정을 확인)")

## 6. 재생 시험
`/workspace/photos` 폴더에 시험용 사진(png/jpg)을 올려 두면 그 사진으로, 없으면 합성 이미지로 돌립니다(합성이면 이벤트는 나지 않고 **속도만** 의미 있음).
사진은 휴대폰 시점 책상 사진이어야 손·물체가 잡힙니다. 응시자 실사진은 올리지 마세요.

In [ ]:
photos = WORK / "photos"
if not (photos.is_dir() and any(photos.iterdir())):
    from PIL import Image, ImageDraw
    photos = WORK / "synthetic"
    photos.mkdir(exist_ok=True)
    img = Image.new("RGB", (512, 384), (200, 200, 200))
    ImageDraw.Draw(img).rectangle((120, 200, 380, 330), fill=(60, 60, 60))
    img.save(photos / "synthetic.jpg")
    print("사진이 없어 합성 이미지 사용:", photos)

r = subprocess.run(
    [sys.executable, "tools/replay/replay.py", "--api-key", API_KEY, "--photos", str(photos),
     "--hold", "30", "--duration", "30", "--settle", "6", "--max-edge", "512"],
    cwd=ROOT, env={**os.environ, "PYTHONPATH": str(ROOT / "src")}, capture_output=True, text=True)
print(r.stdout[-2500:], r.stderr[-800:])

In [ ]:
# 서버 지표: 분석 지연 분포, 처리/폐기 프레임, 이벤트 수, 모델 오류
req = ur.Request("http://localhost:8000/metrics", headers={"Authorization": f"Bearer {API_KEY}"})
text = ur.urlopen(req, timeout=5).read().decode()
for line in text.splitlines():
    if line.startswith(("pair_vision_frame_latency_seconds_bucket", "pair_vision_frames_", "pair_vision_events_total",
                        "pair_vision_detector_errors_total")) and not line.startswith("#"):
        print(line)
print("\n서버 로그에 '프레임 분석 실패' 가 있으면 모델 오류:",
      sum("프레임 분석 실패" in l for l in open(WORK / "vision-server.log")), "건")

## 7. 끝내기
```python
server.terminate()
```
아래 칸을 실행해 서버를 끄고, **RunPod 화면에서 Pod 를 Stop/Terminate** 하세요.

결과로 알려 줄 것: 5번 칸의 `gpu` 값, 6번 칸 출력, 지표의 분석 지연(`frame_latency`) 분포.

In [ ]:
server.terminate()
print("서버 종료")